In [1]:
!git clone https://github.com/boragu/SRL-Code-Review-Study.git

%cd "/content/SRL-Code-Review-Study/Store Sales - Time Series Forecasting/Hyunbin"

!mkdir -p data outputs


Cloning into 'SRL-Code-Review-Study'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 23 (delta 5), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (23/23), 7.36 KiB | 7.36 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/SRL-Code-Review-Study/Store Sales - Time Series Forecasting/Hyunbin


In [2]:
!pip -q install gdown

from pathlib import Path
import zipfile
import gdown

PROJECT_DIR = Path(
    "/content/SRL-Code-Review-Study/"
    "Store Sales - Time Series Forecasting/Hyunbin"
)
DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

ZIP_PATH = DATA_DIR / "store-sales-time-series-forecasting.zip"

gdown.download(
    id="1qUXEalsDiePFDUNuI4ThEihB0Jpr8VEt",
    output=str(ZIP_PATH),
    quiet=False,
)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(DATA_DIR)

print(list(DATA_DIR.rglob("*.csv")))

Downloading...
From: https://drive.google.com/uc?id=1qUXEalsDiePFDUNuI4ThEihB0Jpr8VEt
To: /content/SRL-Code-Review-Study/Store Sales - Time Series Forecasting/Hyunbin/data/store-sales-time-series-forecasting.zip
100%|██████████| 22.4M/22.4M [00:00<00:00, 30.5MB/s]


[PosixPath('/content/SRL-Code-Review-Study/Store Sales - Time Series Forecasting/Hyunbin/data/stores.csv'), PosixPath('/content/SRL-Code-Review-Study/Store Sales - Time Series Forecasting/Hyunbin/data/oil.csv'), PosixPath('/content/SRL-Code-Review-Study/Store Sales - Time Series Forecasting/Hyunbin/data/test.csv'), PosixPath('/content/SRL-Code-Review-Study/Store Sales - Time Series Forecasting/Hyunbin/data/transactions.csv'), PosixPath('/content/SRL-Code-Review-Study/Store Sales - Time Series Forecasting/Hyunbin/data/holidays_events.csv'), PosixPath('/content/SRL-Code-Review-Study/Store Sales - Time Series Forecasting/Hyunbin/data/sample_submission.csv'), PosixPath('/content/SRL-Code-Review-Study/Store Sales - Time Series Forecasting/Hyunbin/data/train.csv')]


In [ ]:
# ==========================================
# Store Sales Forecasting
# ==========================================

import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 장치:", device)

# ------------------------------------------
# 1. 경로 및 데이터 불러오기
# ------------------------------------------
PROJECT_DIR = Path(
    "/content/SRL-Code-Review-Study/"
    "Store Sales - Time Series Forecasting/Hyunbin"
)
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def find_file(filename):
    matches = list(DATA_DIR.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"{filename} 파일을 찾지 못했습니다.")
    return matches[0]

train = pd.read_csv(
    find_file("train.csv"),
    dtype={
        "id": "int32",
        "store_nbr": "int8",
        "family": "category",
        "sales": "float32",
        "onpromotion": "float32",
    },
)

test = pd.read_csv(
    find_file("test.csv"),
    dtype={
        "id": "int32",
        "store_nbr": "int8",
        "family": "category",
        "onpromotion": "float32",
    },
)

stores = pd.read_csv(find_file("stores.csv"))
oil = pd.read_csv(find_file("oil.csv"))
holidays = pd.read_csv(find_file("holidays_events.csv"))

for df in [train, test, oil, holidays]:
    df["date"] = pd.to_datetime(df["date"])

train["onpromotion"] = train["onpromotion"].fillna(0).astype("float32")
test["onpromotion"] = test["onpromotion"].fillna(0).astype("float32")

print("train:", train.shape)
print("test :", test.shape)

# ------------------------------------------
# 2. 기본 특징 생성
# ------------------------------------------
all_families = sorted(
    set(train["family"].astype(str)) | set(test["family"].astype(str))
)
family_to_code = {family: i for i, family in enumerate(all_families)}

for df in [train, test]:
    df["family_code"] = (
        df["family"].astype(str).map(family_to_code).astype("int16")
    )
    df["series_id"] = (
        df["store_nbr"].astype("int32") * 100
        + df["family_code"].astype("int32")
    ).astype("int32")

oil = oil.rename(columns={"dcoilwtico": "oil_price"})

all_dates = pd.date_range(
    min(train["date"].min(), test["date"].min()),
    max(train["date"].max(), test["date"].max()),
    freq="D",
)

oil = (
    oil.set_index("date")
    .reindex(all_dates)
    .rename_axis("date")
    .reset_index()
)

oil["oil_price"] = oil["oil_price"].ffill().bfill().astype("float32")

holidays["transferred"] = (
    holidays["transferred"].astype(str).str.lower().eq("true")
)
holidays = holidays[~holidays["transferred"]].copy()

national_holiday = (
    holidays[holidays["locale"] == "National"]
    .groupby("date")
    .size()
    .clip(upper=1)
)

local_holiday = (
    holidays[holidays["locale"] == "Local"]
    .groupby(["date", "locale_name"])
    .size()
    .clip(upper=1)
)

regional_holiday = (
    holidays[holidays["locale"] == "Regional"]
    .groupby(["date", "locale_name"])
    .size()
    .clip(upper=1)
)

work_day = (
    holidays[holidays["type"] == "Work Day"]
    .groupby("date")
    .size()
    .clip(upper=1)
)

type_to_code = {v: i for i, v in enumerate(sorted(stores["type"].unique()))}
city_to_code = {v: i for i, v in enumerate(sorted(stores["city"].unique()))}
state_to_code = {v: i for i, v in enumerate(sorted(stores["state"].unique()))}

def make_base_features(df):
    df = df.copy()

    df = df.merge(stores, on="store_nbr", how="left")
    df = df.merge(oil, on="date", how="left")

    df["oil_price"] = df["oil_price"].ffill().bfill().astype("float32")

    df["year"] = df["date"].dt.year.astype("int16")
    df["month"] = df["date"].dt.month.astype("int8")
    df["day"] = df["date"].dt.day.astype("int8")
    df["dayofweek"] = df["date"].dt.dayofweek.astype("int8")
    df["weekofyear"] = df["date"].dt.isocalendar().week.astype("int8")
    df["is_weekend"] = (df["dayofweek"] >= 5).astype("int8")

    df["national_holiday"] = (
        df["date"].map(national_holiday).fillna(0).astype("int8")
    )

    local_index = pd.MultiIndex.from_frame(df[["date", "city"]])
    df["local_holiday"] = (
        local_index.map(local_holiday).fillna(0).astype("int8")
    )

    regional_index = pd.MultiIndex.from_frame(df[["date", "state"]])
    df["regional_holiday"] = (
        regional_index.map(regional_holiday).fillna(0).astype("int8")
    )

    df["work_day"] = df["date"].map(work_day).fillna(0).astype("int8")

    df["cluster"] = df["cluster"].fillna(-1).astype("int8")
    df["store_type_code"] = df["type"].map(type_to_code).astype("int8")
    df["city_code"] = df["city"].map(city_to_code).astype("int16")
    df["state_code"] = df["state"].map(state_to_code).astype("int16")

    return df

train_base = make_base_features(train)
test_base = make_base_features(test)

# ------------------------------------------
# 3. 이전 판매량(Lag) 특징 생성
# ------------------------------------------
train_base = train_base.sort_values(["series_id", "date"]).reset_index(drop=True)
train_base["log_sales"] = np.log1p(train_base["sales"]).astype("float32")

for lag in [1, 7, 14, 28]:
    train_base[f"lag_{lag}"] = (
        train_base.groupby("series_id")["log_sales"]
        .shift(lag)
        .astype("float32")
    )

train_base["rolling_mean_7"] = (
    train_base.groupby("series_id")["log_sales"]
    .transform(lambda x: x.shift(1).rolling(7).mean())
    .astype("float32")
)

train_base["rolling_mean_14"] = (
    train_base.groupby("series_id")["log_sales"]
    .transform(lambda x: x.shift(1).rolling(14).mean())
    .astype("float32")
)

model_df = train_base.dropna(
    subset=[
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28",
        "rolling_mean_7",
        "rolling_mean_14",
    ]
).copy()

# ------------------------------------------
# 4. PyTorch 입력 특징 정의
# ------------------------------------------
cat_cols = [
    "store_nbr",
    "family_code",
    "month",
    "dayofweek",
]

num_cols = [
    "cluster",
    "store_type_code",
    "city_code",
    "state_code",
    "onpromotion",
    "oil_price",
    "year",
    "day",
    "weekofyear",
    "is_weekend",
    "national_holiday",
    "local_holiday",
    "regional_holiday",
    "work_day",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
]

valid_dates = sorted(model_df["date"].unique())[-16:]

train_df = model_df[~model_df["date"].isin(valid_dates)].copy()
valid_df = model_df[model_df["date"].isin(valid_dates)].copy()

# 너무 오래 걸릴 경우를 대비한 학습 데이터 상한
MAX_TRAIN_ROWS = 1_500_000

if len(train_df) > MAX_TRAIN_ROWS:
    train_fit_df = train_df.sample(MAX_TRAIN_ROWS, random_state=SEED)
else:
    train_fit_df = train_df.copy()

num_mean = train_fit_df[num_cols].mean()
num_std = train_fit_df[num_cols].std().replace(0, 1)

def make_tensors(df):
    x_cat = df[cat_cols].fillna(0).astype("int64").values

    x_num = (
        (df[num_cols].fillna(0) - num_mean) / num_std
    ).astype("float32").values

    y = df["log_sales"].astype("float32").values

    return (
        torch.tensor(x_cat, dtype=torch.long),
        torch.tensor(x_num, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32),
    )

X_cat_train, X_num_train, y_train = make_tensors(train_fit_df)
X_cat_valid, X_num_valid, y_valid = make_tensors(valid_df)

BATCH_SIZE = 8192

train_loader = DataLoader(
    TensorDataset(X_cat_train, X_num_train, y_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=(device.type == "cuda"),
)

valid_loader = DataLoader(
    TensorDataset(X_cat_valid, X_num_valid, y_valid),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda"),
)

# ------------------------------------------
# 5. PyTorch 신경망 정의
# ------------------------------------------
class SalesForecastNet(nn.Module):
    def __init__(self, num_features):
        super().__init__()

        self.store_embedding = nn.Embedding(55, 8)
        self.family_embedding = nn.Embedding(len(all_families), 8)
        self.month_embedding = nn.Embedding(13, 4)
        self.dayofweek_embedding = nn.Embedding(7, 3)

        embedding_size = 8 + 8 + 4 + 3

        self.network = nn.Sequential(
            nn.Linear(embedding_size + num_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.15),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.10),

            nn.Linear(64, 1),
        )

    def forward(self, x_cat, x_num):
        store_emb = self.store_embedding(x_cat[:, 0])
        family_emb = self.family_embedding(x_cat[:, 1])
        month_emb = self.month_embedding(x_cat[:, 2])
        dayofweek_emb = self.dayofweek_embedding(x_cat[:, 3])

        x = torch.cat(
            [store_emb, family_emb, month_emb, dayofweek_emb, x_num],
            dim=1,
        )

        return self.network(x).squeeze(1)

model = SalesForecastNet(num_features=len(num_cols)).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
)

# ------------------------------------------
# 6. 모델 학습
# ------------------------------------------
EPOCHS = 12
PATIENCE = 4

best_valid_loss = float("inf")
best_state = None
early_stop_count = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum = 0.0

    for x_cat, x_num, y in train_loader:
        x_cat = x_cat.to(device, non_blocking=True)
        x_num = x_num.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()

        pred = model(x_cat, x_num)
        loss = criterion(pred, y)

        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * len(y)

    train_loss = train_loss_sum / len(train_loader.dataset)

    model.eval()
    valid_loss_sum = 0.0

    with torch.no_grad():
        for x_cat, x_num, y in valid_loader:
            x_cat = x_cat.to(device, non_blocking=True)
            x_num = x_num.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            pred = model(x_cat, x_num)
            loss = criterion(pred, y)

            valid_loss_sum += loss.item() * len(y)

    valid_loss = valid_loss_sum / len(valid_loader.dataset)
    valid_rmsle = np.sqrt(valid_loss)

    scheduler.step(valid_loss)

    print(
        f"Epoch {epoch:02d} | "
        f"Train RMSE: {np.sqrt(train_loss):.5f} | "
        f"Valid RMSLE: {valid_rmsle:.5f}"
    )

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }
        early_stop_count = 0
    else:
        early_stop_count += 1

        if early_stop_count >= PATIENCE:
            print("Early stopping")
            break

model.load_state_dict(best_state)
model.eval()

# ------------------------------------------
# 7. 미래 날짜 재귀 예측
# ------------------------------------------
def predict_batch(batch):
    x_cat = torch.tensor(
        batch[cat_cols].fillna(0).astype("int64").values,
        dtype=torch.long,
        device=device,
    )

    x_num = torch.tensor(
        ((batch[num_cols].fillna(0) - num_mean) / num_std)
        .astype("float32")
        .values,
        dtype=torch.float32,
        device=device,
    )

    with torch.no_grad():
        pred_log = model(x_cat, x_num).cpu().numpy()

    return np.maximum(pred_log, 0)

def recursive_forecast(future_base, history_df):
    future_base = future_base.sort_values(["date", "series_id"]).copy()
    predictions = []

    last_logs = (
        history_df.sort_values(["series_id", "date"])
        .groupby("series_id")["log_sales"]
        .apply(lambda x: list(x.tail(28)))
        .to_dict()
    )

    for current_date in sorted(future_base["date"].unique()):
        batch = future_base[future_base["date"] == current_date].copy()

        batch["lag_1"] = [last_logs[sid][-1] for sid in batch["series_id"]]
        batch["lag_7"] = [last_logs[sid][-7] for sid in batch["series_id"]]
        batch["lag_14"] = [last_logs[sid][-14] for sid in batch["series_id"]]
        batch["lag_28"] = [last_logs[sid][-28] for sid in batch["series_id"]]

        batch["rolling_mean_7"] = [
            np.mean(last_logs[sid][-7:]) for sid in batch["series_id"]
        ]
        batch["rolling_mean_14"] = [
            np.mean(last_logs[sid][-14:]) for sid in batch["series_id"]
        ]

        pred_log = predict_batch(batch)

        batch["pred_log_sales"] = pred_log
        predictions.append(batch[["id", "series_id", "pred_log_sales"]])

        for sid, pred in zip(batch["series_id"], pred_log):
            last_logs[sid].append(float(pred))
            last_logs[sid] = last_logs[sid][-28:]

    return pd.concat(predictions, ignore_index=True)

# 마지막 16일 검증 RMSLE
history_for_valid = train_base[train_base["date"] < min(valid_dates)]
valid_base = train_base[train_base["date"].isin(valid_dates)]

valid_predictions = recursive_forecast(valid_base, history_for_valid)

valid_result = valid_df[["id", "log_sales"]].merge(
    valid_predictions[["id", "pred_log_sales"]],
    on="id",
    how="left",
)

recursive_rmsle = np.sqrt(
    np.mean(
        (valid_result["log_sales"] - valid_result["pred_log_sales"]) ** 2
    )
)

print(f"\n재귀 예측 Validation RMSLE: {recursive_rmsle:.5f}")

# ------------------------------------------
# 8. test.csv 예측 및 제출 파일 생성
# ------------------------------------------
test_predictions = recursive_forecast(
    test_base,
    train_base,
)

submission = test_predictions[["id", "pred_log_sales"]].copy()
submission["sales"] = np.maximum(np.expm1(submission["pred_log_sales"]), 0)
submission = submission[["id", "sales"]].sort_values("id")

submission_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)

print("\nsubmission.csv 생성 완료")
print(submission.head())
print(f"\n저장 위치: {submission_path}")

사용 장치: cpu
train: (3000888, 6)
test : (28512, 5)


KeyboardInterrupt: 